In [22]:
# Cell 1: Import Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
import joblib
import os

In [24]:
#Cell 2: Load Dataset
print("Loading dataset...")
data = pd.read_csv('./dataset/Modified_SQL_Dataset.csv')

# Assuming the dataset has 'Query' and 'Label' columns
X = data['Query']
y = data['Label']

print(f"Dataset loaded successfully!")
print(f"Total samples: {len(X)}")
print(f"Safe queries (0): {sum(y == 0)}")
print(f"SQL Injection queries (1): {sum(y == 1)}")

Loading dataset...
Dataset loaded successfully!
Total samples: 30919
Safe queries (0): 19537
SQL Injection queries (1): 11382


In [25]:

# Cell 3: Split Data into Training and Testing Sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"\nData split complete!")
print(f"Training samples: {len(X_train)}")
print(f"Testing samples: {len(X_test)}")


Data split complete!
Training samples: 24735
Testing samples: 6184


In [26]:
# Cell 4: Feature Extraction using TF-IDF
print("\nExtracting features using TF-IDF vectorization...")
vectorizer = TfidfVectorizer()
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print(f"Feature extraction complete!")
print(f"Number of features: {X_train_tfidf.shape[1]}")


Extracting features using TF-IDF vectorization...
Feature extraction complete!
Number of features: 21074


In [27]:
# Cell 5: Train Logistic Regression Model
print("\nTraining Logistic Regression model...")
model = LogisticRegression(max_iter=1000)
model.fit(X_train_tfidf, y_train)

print("Model training complete!")


Training Logistic Regression model...
Model training complete!


In [28]:
# Cell 6: Basic Prediction and Classification Report
y_pred = model.predict(X_test_tfidf)
print("\nBasic Classification Report:")
print(classification_report(y_test, y_pred))


Basic Classification Report:
              precision    recall  f1-score   support

           0       0.96      0.99      0.98      3893
           1       0.99      0.94      0.96      2291

    accuracy                           0.97      6184
   macro avg       0.97      0.96      0.97      6184
weighted avg       0.97      0.97      0.97      6184



In [29]:

# Cell 7: Save Model and Vectorizer
print("\nSaving model and vectorizer...")
os.makedirs('./Models', exist_ok=True)
joblib.dump(model, './Models/sql_injection_model.pkl')
joblib.dump(vectorizer, './Models/vectorizer.pkl')

print(" Model saved: ./Models/sql_injection_model.pkl")
print(" Vectorizer saved: ./Models/vectorizer.pkl")


Saving model and vectorizer...
 Model saved: ./Models/sql_injection_model.pkl
 Vectorizer saved: ./Models/vectorizer.pkl


In [34]:

# Cell 8: COMPREHENSIVE MODEL EVALUATION
print("\n" + "="*70)
print("STARTING COMPREHENSIVE MODEL EVALUATION")
print("="*70)

# Import the evaluator
from model_evaluator import SQLInjectionModelEvaluator

# Initialize evaluator with the trained model
evaluator = SQLInjectionModelEvaluator(
    model_path='./Models/sql_injection_model.pkl',
    vectorizer_path='./Models/vectorizer.pkl'
)

# Run complete evaluation
metrics = evaluator.evaluate_model(
    X_test=X_test, 
    y_test=y_test, 
    save_results=True, 
    output_dir='evaluation_results'
)

print("\n Complete evaluation finished!")
print(" Check 'evaluation_results' folder for:")
print("   - confusion_matrix.png")
print("   - roc_curve.png")
print("   - precision_recall_curve.png")
print("   - metrics_summary.png")
print("   - metrics.json")
print("   - classification_report.txt")


STARTING COMPREHENSIVE MODEL EVALUATION

SQL INJECTION DETECTION MODEL - EVALUATION RESULTS

📊 OVERALL PERFORMANCE METRICS:
Accuracy:                 0.9714 (97.14%)
Precision:                0.9862 (98.62%)
Recall (Sensitivity):     0.9358 (93.58%)
F1-Score:                 0.9604 (96.04%)
Specificity:              0.9923 (99.23%)
ROC AUC Score:            0.9967 (99.67%)
Average Precision:        0.9945 (99.45%)

🔢 CONFUSION MATRIX BREAKDOWN:
True Positives (TP):      2144
True Negatives (TN):      3863
False Positives (FP):     30
False Negatives (FN):     147

📈 DETAILED CLASSIFICATION REPORT:

Class           Precision    Recall       F1-Score     Support     
---------------------------------------------------------------
Safe (0)        0.9633       0.9923       0.9776       3893.0
SQL Injection (1) 0.9862       0.9358       0.9604       2291.0

✅ Confusion matrix saved: evaluation_results/confusion_matrix.png
✅ ROC curve saved: evaluation_results/roc_curve.png
✅ Precision-Reca

In [35]:
# Cell 9: Display Key Metrics
print("\n" + "="*70)
print("KEY PERFORMANCE INDICATORS")
print("="*70)
print(f" Accuracy:      {metrics['accuracy']*100:.2f}%")
print(f" Precision:     {metrics['precision']*100:.2f}%")
print(f" Recall:        {metrics['recall']*100:.2f}%")
print(f" F1-Score:      {metrics['f1_score']*100:.2f}%")
print(f" ROC AUC:       {metrics['roc_auc']*100:.2f}%")


KEY PERFORMANCE INDICATORS
 Accuracy:      97.14%
 Precision:     98.62%
 Recall:        93.58%
 F1-Score:      96.04%
 ROC AUC:       99.67%


In [37]:
#Cell 10: Test with Sample Queries
print("\n" + "="*70)
print("TESTING MODEL WITH SAMPLE QUERIES")
print("="*70)

# Sample test queries
test_queries = [
    "SELECT * FROM users WHERE id = 1",  # Safe
    "SELECT * FROM users WHERE id = 1' OR '1'='1",  # SQL Injection
    "INSERT INTO products (name, price) VALUES ('Phone', 999)",  # Safe
    "'; DROP TABLE users; --",  # SQL Injection
    "SELECT name FROM customers WHERE city = 'New York'"  # Safe
]

print("\nTesting model predictions on sample queries:\n")
for i, query in enumerate(test_queries, 1):
    query_tfidf = vectorizer.transform([query])
    prediction = model.predict(query_tfidf)[0]
    probability = model.predict_proba(query_tfidf)[0]
    
    label = "SQL INJECTION " if prediction == 1 else "SAFE "
    confidence = probability[prediction] * 100
    
    print(f"{i}. Query: {query[:60]}...")
    print(f"   Prediction: {label} (Confidence: {confidence:.2f}%)")
    print()

print("="*70)


TESTING MODEL WITH SAMPLE QUERIES

Testing model predictions on sample queries:

1. Query: SELECT * FROM users WHERE id = 1...
   Prediction: SAFE  (Confidence: 64.41%)

2. Query: SELECT * FROM users WHERE id = 1' OR '1'='1...
   Prediction: SQL INJECTION  (Confidence: 95.90%)

3. Query: INSERT INTO products (name, price) VALUES ('Phone', 999)...
   Prediction: SAFE  (Confidence: 97.57%)

4. Query: '; DROP TABLE users; --...
   Prediction: SQL INJECTION  (Confidence: 55.90%)

5. Query: SELECT name FROM customers WHERE city = 'New York'...
   Prediction: SAFE  (Confidence: 99.06%)

